# Comparar com a versão antiga (exploratório, conversão do projeto)

**Objetivo:** conferir que o fluxo novo (notebooks 01 a 09) gera o mesmo resultado do notebook único antigo (passo 8 da seção 17 do manual).

**Entradas:**
- versão antiga: `03_processamento/versao_antiga/cnpjs_monitor.parquet` (copie para cá o `dados_monitor/cnpjs_monitor.parquet` da pasta antiga);
- versão nova: `04_entrega/cnpjs_monitor.parquet` (notebook 09).

**Como rodar:** antes, rode o fluxo inteiro com `parametros.geocoding.ativo: false` no `config.yaml` (assim o geocoding usa só as respostas já guardadas, como na versão antiga) e depois este notebook. O esperado é **zero diferença**. Diferenças em `lat`, `lon`, `passo`, `metodo`, `confianca` ou `base` aparecem como `ERROR` no log.

**Saída:** só a tabela de diferenças na tela. **Fora do fluxo:** não entra no `executar_tudo.py`.

**Seções:** 1. Carregar as duas versões · 2. CNPJs · 3. Coluna a coluna · 4. Resumo

In [ ]:
# ===================================================================
# PREPARAÇÃO: bibliotecas, config.yaml, log e entradas
# ===================================================================
# Bibliotecas que já vêm com o Python.
import datetime          # datas e horas
import importlib.util    # conferir se uma biblioteca está instalada, sem carregá-la
import json              # ler e gravar arquivos JSON
import os                # trocar arquivos de lugar com segurança
import re                # achar trechos de texto por padrão
import sys               # deixar o Python enxergar a pasta scripts/src
import time              # medir quanto tempo cada etapa leva
from pathlib import Path  # caminhos de pastas e arquivos

# Bibliotecas de cálculos e de tabelas (DataFrames).
import numpy as np
import pandas as pd

# Acha a raiz do projeto (a pasta do config.yaml), subindo a partir da pasta atual,
# e deixa o Python enxergar as funções comuns do projeto (scripts/src).
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())
if str(RAIZ / "scripts") not in sys.path:
    sys.path.insert(0, str(RAIZ / "scripts"))

# Funções comuns: config.yaml, log, validação, arquivos de outras etapas.
from src import config, fontes, registro, validacao
from src.registro import formatar_tempo

# Lê o config.yaml e inicia o log (na tela e em 03_processamento/logs).
cfg = config.carregar(RAIZ)
log = registro.iniciar_log("exploratorio_comparar_versao_antiga", cfg)
# Hora de início do notebook (para o tempo total no resumo final).
inicio_notebook = time.time()
log.info("Preparando o notebook...")

log.info(f"Preparação concluída! ({formatar_tempo(time.time() - inicio_notebook)})")

## 1. Carregar as duas versões

In [ ]:
log.info("SEÇÃO 1: carregando as duas versões...")
# Versão antiga: o Parquet gerado pelo notebook único (antes da conversão), copiado para esta pasta.
ARQ_ANTIGO = cfg.pasta("processamento") / "versao_antiga" / "cnpjs_monitor.parquet"
# Versão nova: o Parquet gerado pelo notebook 09.
ARQ_NOVO = cfg.caminho("entrega.monitor")
for arquivo in (ARQ_ANTIGO, ARQ_NOVO):
    if not arquivo.is_file():
        raise FileNotFoundError(f"Arquivo não encontrado: {arquivo}")
antigo = pd.read_parquet(ARQ_ANTIGO)
novo = pd.read_parquet(ARQ_NOVO)
log.info(f"Antigo: {len(antigo):,} linhas | {antigo.shape[1]} colunas")
log.info(f"Novo:   {len(novo):,} linhas | {novo.shape[1]} colunas")
log.info("SEÇÃO 1 concluída!")

## 2. CNPJs
As duas versões precisam ter os mesmos CNPJs, sem repetição.

In [ ]:
log.info("SEÇÃO 2: conferindo os CNPJs...")
# CNPJ repetido em alguma das versões impediria a comparação linha a linha.
for nome_versao, tabela in (("antigo", antigo), ("novo", novo)):
    repetidos = int(tabela["cnpj"].duplicated().sum())
    (log.info if repetidos == 0 else log.error)(f"CNPJs repetidos na versão {nome_versao}: {repetidos:,}")
# CNPJs que estão só em uma das versões.
so_antigo = set(antigo["cnpj"]) - set(novo["cnpj"])
so_novo = set(novo["cnpj"]) - set(antigo["cnpj"])
(log.info if not so_antigo else log.error)(f"CNPJs só na versão antiga: {len(so_antigo):,}")
(log.info if not so_novo else log.error)(f"CNPJs só na versão nova: {len(so_novo):,}")
# Colunas que estão só em uma das versões.
log.info(f"Colunas só na antiga: {sorted(set(antigo.columns) - set(novo.columns))}")
log.info(f"Colunas só na nova: {sorted(set(novo.columns) - set(antigo.columns))}")
log.info("SEÇÃO 2 concluída!")

## 3. Coluna a coluna
Para cada coluna presente nas duas versões: quantos CNPJs têm valor diferente. Números com diferença acima de 0,000000001; textos, categorias e datas comparados como texto.

In [ ]:
log.info("SEÇÃO 3: comparando coluna a coluna (mesmo CNPJ nas duas versões)...")
# Põe as duas versões na mesma ordem, pelo CNPJ, só com os CNPJs que estão nas duas.
comuns = sorted(set(antigo["cnpj"]) & set(novo["cnpj"]))
a = antigo.drop_duplicates("cnpj").set_index("cnpj").loc[comuns]
b = novo.drop_duplicates("cnpj").set_index("cnpj").loc[comuns]
# Colunas que mais importam: se diferirem, o resultado da cascata mudou.
COLUNAS_CHAVE = ["lat", "lon", "passo", "metodo", "confianca", "base"]
linhas_resultado = []
for coluna in [c for c in a.columns if c in b.columns]:
    x, y = a[coluna], b[coluna]
    if pd.api.types.is_numeric_dtype(x) and pd.api.types.is_numeric_dtype(y):
        # Números: diferença absoluta (vazio nas duas = igual).
        xf, yf = x.astype("float64"), y.astype("float64")
        ambos_vazios = xf.isna() & yf.isna()
        diferentes = ~ambos_vazios & ((xf - yf).abs().fillna(np.inf) > 1e-9)
        maior = float((xf - yf).abs().max()) if diferentes.any() else 0.0
    else:
        # Textos, categorias e datas: comparados como texto (vazio nas duas = igual).
        # Cada valor vira texto; vazio (None, NaN, NaT) vira None, e None == None conta como igual.
        xs = np.array([None if pd.isna(v) else str(v) for v in x.astype("object")], dtype=object)
        ys = np.array([None if pd.isna(v) else str(v) for v in y.astype("object")], dtype=object)
        diferentes = pd.Series(xs != ys, index=x.index)
        maior = None
    n_dif = int(diferentes.sum())
    linhas_resultado.append({"coluna": coluna, "diferencas": n_dif,
                             "percentual": round(n_dif / len(a) * 100, 3) if len(a) else 0.0,
                             "maior_diferenca_numerica": maior,
                             "tipo_antigo": str(x.dtype), "tipo_novo": str(y.dtype)})
    if n_dif:
        nivel = log.error if coluna in COLUNAS_CHAVE else log.warning
        nivel(f"    {coluna}: {n_dif:,} diferenças ({n_dif / len(a) * 100:.3f}%)")
comparacao = pd.DataFrame(linhas_resultado).sort_values("diferencas", ascending=False)
log.info("SEÇÃO 3 concluída!")
display(comparacao)

## 4. Resumo

In [ ]:
log.info("RESUMO da comparação...")
total_dif = int(comparacao["diferencas"].sum())
dif_chave = int(comparacao.loc[comparacao["coluna"].isin(COLUNAS_CHAVE), "diferencas"].sum())
if total_dif == 0 and not so_antigo and not so_novo:
    log.info("As duas versões são IGUAIS: a conversão não mudou o resultado!")
else:
    log.warning(f"Diferenças: {total_dif:,} no total, {dif_chave:,} nas colunas-chave {COLUNAS_CHAVE}. "
                "Confira as colunas acima antes de apagar a versão antiga.")
log.info(f"RESUMO concluído! Tempo total: {formatar_tempo(time.time() - inicio_notebook)}.")